# Visual Geometry Grounded Transformer
VGGT (Meta + Oxford VGG, CVPR 2025 Best Paper) memprediksi **pose kamera, intrinsik, depth map, point map, dan 3D track** dari 1 sampai ratusan gambar dalam satu forward pass

In [ ]:
# import os; os._exit(0)

: 

## 1. Setup

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 3080, 10240 MiB


In [7]:
import os
if not os.path.exists("vggt"):
    !git clone -q https://github.com/facebookresearch/vggt.git
%cd vggt
%pip install -r requirements.txt -e . plotly trimesh matplotlib ipywidgets

/home/alpha/vggt/vggt
Obtaining file:///home/alpha/vggt/vggt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vggt (pyproject.toml) ... done
  Created wheel for vggt: filename=vggt-0.0.1-0.editable-py3-none-any.whl size=7296 sha256=9e3520f48ea37285d39e0910c97dbd52d2dee55ca99314b017d2a25c7c755eb8
  Stored in directory: /tmp/pip-ephem-wheel-cache-9jczie93/wheels/c1/02/af/7194adf41bcfa2244e3afffe56c2eaa7bd5be1172a2e1a1e9e
Successfully built vggt
  Attempting uninstall: vggt
    Found existing installation: vggt 0.0.1
    Uninstalling vggt-0.0.1:
      Successfully uninstalled vggt-0.0.1
Note: you may need to restart the kernel to use updated packages.


plotly: 3D interactive plot (kamera dan point cloud) \
trimesh: menyimpan point cloud ke file .ply \
matplotlib: menampilkan gambar, depth map, dan tracking \
opencv-python-headless: memotong video jadi frame. Versi headless dipakai karena Colab tidak punya layar. 

## 2. Load model
Bobot (~5 GB) otomatis diunduh dari Hugging Face saat pertama kali. `facebook/VGGT-1B` lisensinya non-komersial; untuk komersial pakai `facebook/VGGT-1B-Commercial` (perlu isi form akses di HF).

In [4]:
import torch, numpy as np, glob, time
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    # bfloat16 butuh GPU Ampere+ (A100, L4, RTX 30xx). T4 pakai float16.
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    dtype = torch.float32
print(device, dtype)

model = VGGT.from_pretrained("facebook/VGGT-1B").to(device).eval()
print(f"Parameter: {sum(p.numel() for p in model.parameters())/1e9:.2f} B")

cuda torch.bfloat16


Parameter: 1.26 B


mengapa .eval() ? 
1. matikan droupout -> krn kita cuma butuh untuk inference only
2. BatchNorm dibebukan

load_and_preprocess_images: membaca foto lalu mengubah ukurannya ke lebar 518 px dan nilai piksel ke 0–1, sesuai format yang diharapkan model. \
pose_encoding_to_extri_intri: model mengeluarkan pose kamera dalam bentuk kode ringkas (9 angka). Fungsi ini mengubahnya jadi matriks kamera standar (extrinsic dan intrinsic). \
unproject_depth_map_to_point_map: mengubah depth map + data kamera menjadi titik-titik 3D.

## 3. Pilih gambar
Tiga opsi — pilih **salah satu** dengan mengatur `SOURCE`:
- `"example"` → scene contoh dari folder `examples/` di repo
- `"upload"` → upload foto sendiri (beberapa foto objek/ruangan yang sama dari sudut berbeda)
- `"video"` → upload video, ambil frame tiap N

In [ ]:
import os, glob

SOURCE     = "arca"
MAX_FRAMES = 16

if SOURCE == "example":
    scenes = sorted(glob.glob("examples/*/images"))
    print("Scene tersedia:", [s.split("/")[1] for s in scenes])
    scene = scenes[0]                          # ganti angkanya untuk scene lain
    image_names = sorted(glob.glob(os.path.join(scene, "*")))

elif SOURCE != "example":
    FOLDER = "/mnt/c/Users/Alphasm/Pictures/vggt"
    image_names = sorted(glob.glob(os.path.join(FOLDER, "*")))

image_names = image_names[:MAX_FRAMES]
print(len(image_names), "gambar:")
for p in image_names:
    print("  ", p)

0 gambar:


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
n = len(image_names); cols = min(n, 6); rows = (n + cols - 1) // cols
fig, axs = plt.subplots(rows, cols, figsize=(3*cols, 3*rows), squeeze=False)
for ax in axs.flat: ax.axis("off")
for ax, p in zip(axs.flat, image_names):
    ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p), fontsize=8)
plt.tight_layout(); plt.show()

## 4. Inference
`load_and_preprocess_images` me-resize ke lebar 518 px (tinggi kelipatan 14) dengan nilai [0,1]. Satu panggilan `model(images)` menjalankan semua head.

In [ ]:
images = load_and_preprocess_images(image_names).to(device)   # (S, 3, H, W)
print("images:", tuple(images.shape))

if device == "cuda":
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
t0 = time.time()
with torch.no_grad(), torch.autocast(device_type=device, dtype=dtype, enabled=(device=="cuda")):
    predictions = model(images)
if device == "cuda": torch.cuda.synchronize()
print(f"Waktu inference: {time.time()-t0:.2f} s")
if device == "cuda":
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

for k, v in predictions.items():
    if torch.is_tensor(v): print(f"{k:20s}", tuple(v.shape))

In [ ]:
# Pose encoding -> matriks kamera (konvensi OpenCV, camera-from-world)
extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions["pose_enc"], images.shape[-2:])

to_np = lambda t: t.float().squeeze(0).cpu().numpy()
E   = to_np(extrinsic)                      # (S, 3, 4)
K   = to_np(intrinsic)                      # (S, 3, 3)
D   = to_np(predictions["depth"])[..., 0]   # (S, H, W)
Dc  = to_np(predictions["depth_conf"])      # (S, H, W)
WP  = to_np(predictions["world_points"])    # (S, H, W, 3)
WPc = to_np(predictions["world_points_conf"])
IMG = images.permute(0, 2, 3, 1).cpu().numpy()   # (S, H, W, 3)
S, H, W = D.shape
print("Frame pertama jadi referensi dunia:\n", np.round(E[0], 3))
print("Intrinsik frame 0:\n", np.round(K[0], 1))

## 5. Kamera
Pusat kamera di koordinat dunia: $C = -R^\top t$. Panah menunjukkan arah pandang (+z kamera).

In [ ]:
import plotly.graph_objects as go

R, t = E[:, :, :3], E[:, :, 3]
centers = -np.einsum("sji,sj->si", R, t)
forward = R[:, 2, :]                # sumbu z kamera dalam koordinat dunia
scale = np.linalg.norm(centers.max(0) - centers.min(0)) * 0.15 + 1e-3

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=centers[:,0], y=centers[:,1], z=centers[:,2], mode="markers+text+lines",
                           text=[str(i) for i in range(S)], marker=dict(size=4, color="red"), name="kamera"))
for c, f in zip(centers, forward):
    e = c + f * scale
    fig.add_trace(go.Scatter3d(x=[c[0], e[0]], y=[c[1], e[1]], z=[c[2], e[2]], mode="lines",
                               line=dict(color="blue", width=4), showlegend=False))
fig.update_layout(scene=dict(aspectmode="data"), height=550, title="Trajektori kamera")
fig.show()

fov = np.degrees(2 * np.arctan(W / (2 * K[:, 0, 0])))
print("FoV horizontal per frame (derajat):", np.round(fov, 1))

## 6. Depth map + confidence
Depth VGGT bersifat relatif (skala dinormalisasi), bukan meter.

In [ ]:
show = min(S, 4)
fig, axs = plt.subplots(3, show, figsize=(4*show, 9), squeeze=False)
for i in range(show):
    axs[0, i].imshow(IMG[i]);                     axs[0, i].set_title(f"frame {i}")
    axs[1, i].imshow(D[i], cmap="turbo");         axs[1, i].set_title("depth")
    axs[2, i].imshow(np.log(Dc[i]), cmap="magma"); axs[2, i].set_title("log confidence")
for ax in axs.flat: ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Point cloud
Dua cara mendapat titik 3D:
- **Depth + kamera → unproject** (biasanya lebih akurat menurut penulis)
- **Point-map head** langsung

Titik disaring berdasarkan persentil confidence. Coba ubah `CONF_PERCENTILE`.

In [ ]:
CONF_PERCENTILE = 50     # buang 50% titik paling tidak yakin
MAX_POINTS = 150_000

pts_depth = unproject_depth_map_to_point_map(predictions["depth"].squeeze(0).float().cpu().numpy(), E, K)  # (S,H,W,3)

def make_cloud(points, conf):
    p, c, col = points.reshape(-1, 3), conf.reshape(-1), IMG.reshape(-1, 3)
    keep = (c >= np.percentile(c, CONF_PERCENTILE)) & np.isfinite(p).all(1)
    p, col = p[keep], col[keep]
    if len(p) > MAX_POINTS:
        idx = np.random.choice(len(p), MAX_POINTS, replace=False); p, col = p[idx], col[idx]
    return p, col

def plot_cloud(p, col, title):
    rgb = [f"rgb({r},{g},{b})" for r, g, b in (col * 255).astype(int)]
    fig = go.Figure(go.Scatter3d(x=p[:,0], y=-p[:,1], z=-p[:,2], mode="markers",
                                 marker=dict(size=1, color=rgb)))
    fig.add_trace(go.Scatter3d(x=centers[:,0], y=-centers[:,1], z=-centers[:,2], mode="markers",
                               marker=dict(size=4, color="red"), name="kamera"))
    fig.update_layout(scene=dict(aspectmode="data"), height=650, title=title, showlegend=False)
    fig.show()

p_d, c_d = make_cloud(pts_depth, Dc)
p_m, c_m = make_cloud(WP, WPc)
plot_cloud(p_d, c_d, f"Dari depth + kamera ({len(p_d):,} titik)")

In [ ]:
plot_cloud(p_m, c_m, f"Dari point-map head ({len(p_m):,} titik)")

# Seberapa beda dua cabang ini?
diff = np.linalg.norm(pts_depth - WP, axis=-1)
print(f"Median jarak depth-vs-pointmap: {np.median(diff):.4f}  (skala scene ~{np.linalg.norm(p_d.std(0)):.3f})")

In [ ]:
# Ekspor .ply — buka di MeshLab / CloudCompare / Blender
import trimesh
trimesh.PointCloud(p_d, colors=(c_d * 255).astype(np.uint8)).export("/content/vggt_cloud.ply")
print("Tersimpan: /content/vggt_cloud.ply")
# from google.colab import files; files.download("/content/vggt_cloud.ply")

## 8. Tracking titik
Pilih titik query di **frame 0** (koordinat piksel pada gambar ter-resize, ukuran W×H di atas). Track head mengikuti titik itu ke semua frame.

In [ ]:
print(f"Ukuran gambar: {W} x {H}")
# Grid titik otomatis; ganti dengan koordinat pilihanmu: [[x, y], ...]
gx, gy = np.meshgrid(np.linspace(0.2*W, 0.8*W, 5), np.linspace(0.2*H, 0.8*H, 4))
query = torch.tensor(np.stack([gx.ravel(), gy.ravel()], 1), dtype=torch.float32, device=device)

with torch.no_grad():
    with torch.autocast(device_type=device, dtype=dtype, enabled=(device=="cuda")):
        tokens, ps_idx = model.aggregator(images[None])
    track_list, vis, conf = model.track_head(tokens, images[None], ps_idx, query_points=query[None])

tracks = track_list[-1].squeeze(0).float().cpu().numpy()     # (S, N, 2)
vis = vis.squeeze(0).float().cpu().numpy()                    # (S, N)
print("tracks:", tracks.shape)

In [ ]:
show = min(S, 6)
colors = plt.cm.hsv(np.linspace(0, 1, tracks.shape[1], endpoint=False))
fig, axs = plt.subplots(1, show, figsize=(4*show, 4), squeeze=False)
for i in range(show):
    ax = axs[0, i]; ax.imshow(IMG[i]); ax.axis("off"); ax.set_title(f"frame {i}")
    v = vis[i] > 0.5
    ax.scatter(tracks[i, v, 0], tracks[i, v, 1], c=colors[v], s=30, edgecolors="white")
    ax.scatter(tracks[i, ~v, 0], tracks[i, ~v, 1], c=colors[~v], s=30, marker="x")
plt.suptitle("Bulat = terlihat, x = tertutup/di luar frame"); plt.tight_layout(); plt.show()

## 9. Ide eksperimen lanjutan
Jalankan ulang bagian 3–7 dengan variasi berikut:

### 9a. Single-view depth
VGGT tidak dilatih khusus untuk monokular, tapi hasilnya kompetitif. Cukup beri satu gambar.

In [ ]:
single = load_and_preprocess_images(image_names[:1]).to(device)
with torch.no_grad(), torch.autocast(device_type=device, dtype=dtype, enabled=(device=="cuda")):
    pred1 = model(single)
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].imshow(IMG[0]); axs[0].set_title("input")
axs[1].imshow(pred1["depth"][0, 0, ..., 0].float().cpu(), cmap="turbo"); axs[1].set_title("depth (1 view)")
axs[2].imshow(D[0], cmap="turbo"); axs[2].set_title(f"depth ({S} views)")
for a in axs: a.axis("off")
plt.show()

### 9b. Skala jumlah frame → waktu & VRAM

In [ ]:
results = []
for n in [1, 2, 4, 8, 16, 32]:
    if n > len(image_names): break
    imgs = load_and_preprocess_images(image_names[:n]).to(device)
    if device == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad(), torch.autocast(device_type=device, dtype=dtype, enabled=(device=="cuda")):
        _ = model(imgs)
    if device == "cuda": torch.cuda.synchronize()
    mem = torch.cuda.max_memory_allocated()/1e9 if device == "cuda" else float("nan")
    results.append((n, time.time()-t0, mem)); print(f"{n:3d} frame: {results[-1][1]:.2f}s, {mem:.2f} GB")
    del _, imgs

### 9c. Masking area yang mengganggu
Piksel yang tidak diinginkan (langit, kaca, air, objek bergerak) bisa di-mask dengan mengisinya 0 atau 1 — kotak kasar pun cukup.

In [ ]:
masked = images.clone()
masked[:, :, : H // 4, :] = 0.0     # contoh: hitamkan seperempat atas (mis. langit)
with torch.no_grad(), torch.autocast(device_type=device, dtype=dtype, enabled=(device=="cuda")):
    pred_m = model(masked)
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].imshow(masked[0].permute(1,2,0).cpu()); axs[0].set_title("input ter-mask")
axs[1].imshow(D[0], cmap="turbo"); axs[1].set_title("depth asli")
axs[2].imshow(pred_m["depth"][0, 0, ..., 0].float().cpu(), cmap="turbo"); axs[2].set_title("depth ter-mask")
for a in axs: a.axis("off")
plt.show()

### 9d. Ekspor ke COLMAP (untuk Gaussian Splatting / NeRF)
Script repo butuh struktur `SCENE_DIR/images/`. Hasil disimpan di `SCENE_DIR/sparse/`, bisa langsung dipakai `gsplat`.

In [ ]:
import shutil
SCENE_DIR = "/content/colmap_scene"
os.makedirs(f"{SCENE_DIR}/images", exist_ok=True)
for p in image_names: shutil.copy(p, f"{SCENE_DIR}/images/")

!python demo_colmap.py --scene_dir={SCENE_DIR}
# Dengan bundle adjustment (lebih akurat, lebih lama; mungkin perlu: pip install pycolmap lightglue):
# !python demo_colmap.py --scene_dir={SCENE_DIR} --use_ba --max_query_pts=2048 --query_frame_num=5
!ls -la {SCENE_DIR}/sparse

---
**Tips**
- OOM? Kurangi `MAX_FRAMES`, atau pakai GPU yang lebih besar.
- Hasil bagus butuh overlap antar-gambar yang cukup; foto mengelilingi objek bekerja paling baik.
- Demo interaktif resmi: `python demo_gradio.py` (perlu `pip install -r requirements_demo.txt`) atau [HF Space](https://huggingface.co/spaces/facebook/vggt).